# Demo A — public 10×U-Net ensemble with 8-view TTA (final submission, public LB 0.39)

Reproduces Kaggle submission ref 57768486 (public PQ 0.39; 1,177 instances over 177 of the 180 test images).

Inputs this notebook needs (both public, no challenge)
1. the competition data (attach this competition): test/test_images/*.jpeg and the training annotations;
2. the public weights dataset dariushafshar/solar-filament-unet-weights (10 checkpoints, solar_unet_receipt.json,
   folds.csv, and a segmentation-models-pytorch wheel). Third-party weights keep their published licence.

Pipeline shown: preprocessing (2048 canvas, 8 dihedral TTA views) -> inference with the published
checkpoints (no training required; training code for our own cascade is in src/ and docs/TRAIN.md) ->
postprocessing (probability averaging, KEEP rule t_lo=0.35 / t_hi=0.99 / min area 200 px / mean prob >= 0.85,
per-image pixel-disjoint assembly) -> CSV validation -> pooled-PQ evaluation.

Runtime about 20 min on one T4. Our code is MIT; checkpoints keep their licence.

In [ ]:
import glob, os
from pathlib import Path

REPO = Path.cwd()          # repository root (this notebook lives in repo/notebooks)
INPUT = Path('/kaggle/input')

receipts = glob.glob(str(INPUT / '**' / 'solar_unet_receipt.json'), recursive=True)
test_dirs = [p for p in glob.glob(str(INPUT / '**' / 'test_images'), recursive=True) if os.path.isdir(p)]
train_json = glob.glob(str(INPUT / '**' / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'), recursive=True)

print('repo:', REPO)
print('weights receipt:', receipts[:1])
print('test_images:', test_dirs)
print('train annotations:', train_json[:1])

assert receipts, 'attach the public weights dataset dariushafshar/solar-filament-unet-weights'
assert test_dirs, 'attach this competition (test/test_images)'

In [ ]:
# Inference with the published checkpoints (the script is Kaggle-path aware: /kaggle/input -> submission.csv).
# Edit CONFIG at the top of the script to trade quality for time: max_models=2, tta=2 for a format smoke test.
!python scripts/infer_dariush_ensemble.py

In [ ]:
# Validate the produced submission (schema, RLE, per-image pixel-disjointness) and summarise it.
!python src/validate_submission.py submission.csv

import pandas as pd
df = pd.read_csv('submission.csv')
stems = df['filament_id'].str.rsplit('_', n=1).str[0]
print('rows:', len(df), '| images:', stems.nunique())

In [ ]:
# Evaluation on honest out-of-fold predictions with the organisers' pooled PQ, plus the Dice/IoU
# distributions and one-to-many / many-to-one counts required by the rubric.
# assets/dariush_fold0_oof.csv ships the fold-0 out-of-fold predictions of this same pipeline.
gt = (glob.glob(str(INPUT / '**' / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'), recursive=True) or [''])[0]
!python src/eval_demo.py --pred assets/dariush_fold0_oof.csv --gt-json "$gt" --stems-json assets/fold0_images.json

Expected numbers (verified in this repository)

| quantity | value |
|---|---|
| test submission | 1,177 instances / 177 images, pixel-disjoint, public LB 0.39 |
| out-of-fold PQ (fold 0, 232 annotator records) | 0.4755 (SQ 0.6881, RQ 0.6910; TP/FP/FN = 1238/528/579) |
| Dice over non-zero pairs | mean 0.734 (median 0.791) |
| IoU over non-zero pairs | mean 0.605 (median 0.655) |
| one-to-many / many-to-one | 84 annotations and 44 predictions overlap 2-3 masks; 305 annotations / 209 predictions unmatched |

With only 2 models and TTA 2 (ref 57767915) the same code path scores 0.37, documenting the ensemble's strength.